In [ ]:
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

# Student Identity Inputs
LAST_NAME = "Asis"
STUDENT_ID = "TUPM-26-1217"
SEED_DIGIT = int(STUDENT_ID[-1])  # 7

# Data Generation Rules
TARGET_YEAR = 2015 + SEED_DIGIT  # 2022

# File Paths
DATASET_PATH = Path("data/Global_Cybersecurity_Threats_2015-2024.csv")
FILTERED_CSV_PATH = Path("output/filtered_cybersecurity_2022.csv")
SUMMARY_JSON_PATH = Path("output/cybersecurity_summary_2022.json")
BACKUP_JSON_PATH = Path("backup/cybersecurity_summary_2022_backup.json")
REPORT_TXT_PATH = Path("output/cybersecurity_report_2022.txt")

# Data Containers
total_dataset_records = 0
target_year_incidents = []
attack_type_counts = Counter()
total_financial_loss = 0.0

# 1. Read CSV line-by-line and parse records
with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_dataset_records += 1

        try:
            raw_timestamp = row.get("Timestamp", row.get("Year", "")).strip()
            year_str = raw_timestamp.split("-")[0].strip()

            if not year_str.isdigit():
                continue

            year_val = int(year_str)

            if year_val == TARGET_YEAR:
                attack_type = row.get("Attack Type", "").strip()
                loss_str = row.get("Financial Loss (in Million $)", "").strip()

                if not loss_str:
                    continue

                loss_val = float(loss_str)

                record = {
                    "Timestamp": raw_timestamp,
                    "Attack Type": attack_type,
                    "Financial Loss (in Million $)": loss_val,
                }

                target_year_incidents.append(record)
                attack_type_counts[attack_type] += 1
                total_financial_loss += loss_val

        except (ValueError, TypeError):
            continue

# Compute summary statistics
incidents_count = len(target_year_incidents)
avg_financial_loss = (total_financial_loss / incidents_count) if incidents_count > 0 else 0.0

top_5_costliest = sorted(
    target_year_incidents,
    key=lambda x: x["Financial Loss (in Million $)"],
    reverse=True
)[:5]

# Export filtered records to CSV
fieldnames = ["Timestamp", "Attack Type", "Financial Loss (in Million $)"]
with open(FILTERED_CSV_PATH, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(target_year_incidents)

# Export JSON summary & create verified backup
summary_data = {
    "student_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "target_year": TARGET_YEAR,
    "total_records_in_dataset": total_dataset_records,
    "incidents_in_target_year": incidents_count,
    "count_per_attack_type": dict(attack_type_counts),
    "total_financial_loss_usd_millions": round(total_financial_loss, 2),
    "average_financial_loss_usd_millions": round(avg_financial_loss, 2),
}

with open(SUMMARY_JSON_PATH, mode="w", encoding="utf-8") as file:
    json.dump(summary_data, file, indent=4)

shutil.copy(SUMMARY_JSON_PATH, BACKUP_JSON_PATH)
backup_verified = BACKUP_JSON_PATH.exists()

# Create text report
with open(REPORT_TXT_PATH, mode="w", encoding="utf-8") as file:
    file.write("EXERCISE 3: CYBERSECURITY INCIDENT REPORT\n")
    file.write("=" * 70 + "\n")
    file.write(f"Student Name: {LAST_NAME}\n")
    file.write(f"Student ID: {STUDENT_ID}\n")
    file.write(f"Target Year: {TARGET_YEAR}\n")
    file.write(f"Total Dataset Records: {total_dataset_records}\n")
    file.write(f"Incidents in Target Year: {incidents_count}\n")
    file.write(f"Attack Type Counts: {dict(attack_type_counts)}\n")
    file.write(f"Total Financial Loss: ${total_financial_loss:,.2f} Million\n")
    file.write(f"Average Financial Loss: ${avg_financial_loss:,.2f} Million\n")
    file.write("\nFive Costliest Incidents:\n")

    for idx, inc in enumerate(top_5_costliest, start=1):
        loss = inc.get("Financial Loss (in Million $)", 0.0)
        atk = inc.get("Attack Type", "Unknown")
        file.write(f"{idx}. Loss: ${loss:.2f}M | Type: {atk}\n")

print("=" * 70)
print(f"{'EXERCISE 3: CYBERSECURITY INCIDENT REPORT':^70}")
print("=" * 70)
print(f"Target Year            : {TARGET_YEAR}")
print(f"Total Dataset Records  : {total_dataset_records}")
print(f"Incidents in {TARGET_YEAR}     : {incidents_count}")
print(f"Attack Type Counts     : {dict(attack_type_counts)}")
print(f"Total Financial Loss   : ${total_financial_loss:,.2f} Million")
print(f"Avg Financial Loss     : ${avg_financial_loss:,.2f} Million")
print("-" * 70)
print("Five Costliest Incidents:")
for idx, inc in enumerate(top_5_costliest, start=1):
    inc_id = inc.get("Incident ID", "N/A")
    loss = inc.get("Financial Loss (in Million $)", 0.0)
    atk = inc.get("Attack Type", "Unknown")
    print(f"  {idx}. Loss: ${loss:>6.2f}M | Type: {atk}")
print("-" * 70)
print(f"Filtered CSV File      : {FILTERED_CSV_PATH.name}")
print(f"JSON Summary File      : {SUMMARY_JSON_PATH.name}")
print(f"Backup Verification    : {'VERIFIED' if backup_verified else 'FAILED'}")
print(f"Text Report File       : {REPORT_TXT_PATH.name}")
print("=" * 70)

              EXERCISE 3: CYBERSECURITY INCIDENT REPORT               
Target Year            : 2022
Total Dataset Records  : 3000
Incidents in 2022     : 318
Attack Type Counts     : {'Phishing': 57, 'Malware': 64, 'Ransomware': 56, 'SQL Injection': 46, 'DDoS': 54, 'Man-in-the-Middle': 41}
Total Financial Loss   : $15,870.86 Million
Avg Financial Loss     : $49.91 Million
----------------------------------------------------------------------
Five Costliest Incidents:
  1. Loss: $ 98.47M | Type: Phishing
  2. Loss: $ 98.46M | Type: SQL Injection
  3. Loss: $ 98.06M | Type: Phishing
  4. Loss: $ 97.56M | Type: Phishing
  5. Loss: $ 97.32M | Type: Malware
----------------------------------------------------------------------
Filtered CSV File      : filtered_cybersecurity_2022.csv
JSON Summary File      : cybersecurity_summary_2022.json
Backup Verification    : VERIFIED
Text Report File       : cybersecurity_report_2022.txt
